# model-5: MobileNetV3Large

Notebook untuk model-5 (MobileNetV3Large).
Versi 2 (8 November 2026): data dibaca dari dataset Kaggle `simplicity-224-letterbox`.

**Pengaturan Kaggle (wajib):**
- *Settings > Accelerator*: GPU (T4 atau P100)
- *Settings > Internet*: **ON**, untuk `git clone` dan unduh bobot ImageNet (butuh verifikasi nomor HP)
- *Add Input* (panel kanan): cari dataset `simplicity-224-letterbox` milik `yonobengkel12`, lalu tambahkan. Isinya gambar 224×224 yang sudah jadi dan identik dengan hasil `data_utils.py`, jadi semua pasangan memakai input yang persis sama.
- Untuk pelatihan panjang, pakai *Save Version > Save & Run All* agar tetap berjalan walau browser ditutup.

**Aturan:** sel bertanda `JANGAN DIUBAH` dipakai apa adanya. `common/data_utils.py` dikunci dan hanya dipakai lewat import.

## 1. Setup bersama (JANGAN DIUBAH)
Clone repo (di Kaggle) atau cari root repo (di lokal), import `data_utils`, cetak sidik versinya, lalu cek perangkat.
Sidik yang tercetak harus sama dengan sidik resmi di README bagian 3 (per 8 November 2026: `b3e0629783c5`).

In [ ]:
# ===== SETUP BERSAMA: JANGAN DIUBAH =====
import subprocess
import sys
from pathlib import Path
REPO_URL = "https://github.com/0Raja0-RA/simplicity-cnn-comparison.git"
BRANCH = "main"
if Path("/kaggle/working").exists():                      # di Kaggle
    ROOT = Path("/kaggle/working/simplicity-cnn-comparison")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
else:                                                     # di lokal: naik sampai ketemu root repo
    ROOT = Path.cwd().resolve()
    while not (ROOT / "common" / "data_utils.py").exists():
        if ROOT.parent == ROOT:
            raise FileNotFoundError("Root repo (berisi common/data_utils.py) tidak ditemukan.")
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from common import data_utils as du
SIDIK = du.sidik_config()
INFO_PERANGKAT = du.cek_perangkat()
print("ROOT:", ROOT)


## 2. Data (JANGAN DIUBAH, kecuali `BATCH_SIZE`)
Gambar float32 0-255 berukuran `du.IMG_SIZE`, sudah di-padding, belum dinormalisasi. Label integer 0-9
(pakai `sparse_categorical_crossentropy`). Augmentasi **tidak** ada di dataset; pasang `du.get_augmentation()` di model.
`du.load_data()` mencetak sumber data. Di Kaggle harus tertulis `simplicity_224.npz (npz dataset Kaggle)`; sel ini berhenti dengan pesan galat kalau dataset belum dipasang.

In [ ]:
BATCH_SIZE = 32          # boleh diubah per pasangan (README bagian 5)
# Di Kaggle, dataset simplicity-224-letterbox wajib terpasang (Add Input) supaya semua pasangan memakai gambar yang sama persis
if Path("/kaggle/working").exists() and not any(Path("/kaggle/input").rglob(du.NPZ_NAMA)):
    raise RuntimeError(f"Dataset Kaggle {du.KAGGLE_DATASET} belum dipasang. "
                       "Tambahkan lewat Add Input di panel kanan, lalu jalankan ulang sel ini.")
train_ds, val_ds = du.load_data(batch_size=BATCH_SIZE)   # mencetak sumber data: harus simplicity_224.npz
CLASS_NAMES = du.get_class_names()
print("IMG_SIZE:", du.IMG_SIZE, "| kelas:", CLASS_NAMES)


## 3. Model (diisi pasangan)
Urutan wajib di dalam model:
`Input(IMG_SIZE + (3,))` -> `du.get_augmentation()` -> `preprocess_input` milik arsitektur -> backbone ImageNet
(`include_top=False`) -> head 10 kelas softmax.

Arsitektur: `keras.applications.MobileNetV3Large`
Parameter Keras yang dieksplorasi:
- `alpha`: width multiplier (misal 1.0 vs 0.75)
- `minimalistic`: varian tanpa SE block dan hard-swish (CPU-friendly) vs varian lengkap
- `pooling`: 'avg' vs 'max'
- Normalisasi: `keras.applications.mobilenet_v3.preprocess_input` sesudah augmentasi.

In [ ]:
from tensorflow import keras
def bangun_model(
    alpha=1.0,
    minimalistic=False,
    pooling="avg",
    unfreeze_pct=0,
    dropout=0.2
):
    inputs = keras.Input(shape=du.IMG_SIZE + (3,))
    x = du.get_augmentation()(inputs)                       # wajib, aktif hanya saat training
    
    # Di Keras 3, include_preprocessing=True (default) secara otomatis menyertakan
    # layer Rescaling(scale=1/127.5, offset=-1.0) untuk menormalisasi piksel [0, 255] ke [-1, 1].
    base = keras.applications.MobileNetV3Large(
        input_shape=du.IMG_SIZE + (3,),
        alpha=alpha,
        minimalistic=minimalistic,
        include_top=False,
        weights="imagenet",
        pooling=pooling,
        include_preprocessing=True
    )
    
    if unfreeze_pct <= 0:
        base.trainable = False
    else:
        base.trainable = True
        num_layers = len(base.layers)
        cutoff = int(num_layers * (1.0 - (unfreeze_pct / 100.0)))
        for layer in base.layers[:cutoff]:
            layer.trainable = False
        # BatchNormalization tetap dibekukan agar statistik ImageNet stabil
        for layer in base.layers[cutoff:]:
            if isinstance(layer, keras.layers.BatchNormalization):
                layer.trainable = False
                
    x = base(x, training=False)
    x = keras.layers.Dropout(dropout)(x)
    outputs = keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
    return keras.Model(inputs, outputs)


## 4. Tuning: random search (diisi pasangan)
Ikuti README bagian 5: jumlah 10 trial, epoch maksimum 40, dan early stopping (patience 8).
Catat setiap trial ke `trials.csv` langsung setiap kali selesai satu run.
Pilih konfigurasi terbaik dari **validation** saja. Kalau seri, pilih val_loss terendah.

### Catatan Perbedaan Tiap Percobaan (Trial 1 - 10):
- **Trial 1 (Baseline):** `alpha=1.0, mini=False, pool=avg, lr=1e-3, opt=adam, freeze=100%, bs=32, drop=0.2` (titik acuan fitur ImageNet default)
- **Trial 2:** `lr=5e-4, drop=0.3` (mengecilkan LR & menaikkan dropout untuk stabilisasi konvergensi)
- **Trial 3:** `alpha=0.75` (menguji width multiplier lebih kecil, menguji apakah model lebih ramping mencegah overfit)
- **Trial 4:** `minimalistic=True` (mematikan modul SE attention dan Hard-Swish untuk menguji kontribusinya)
- **Trial 5:** `pooling=max` (menguji GlobalMaxPooling vs GlobalAveragePooling pada aktivasi spasial)
- **Trial 6:** `unfreeze=20%, lr=1e-4, drop=0.3` (fine-tuning 20% layer konvolusi teratas dengan LR kecil)
- **Trial 7:** `opt=adamw, bs=16, lr=3e-4` (menguji optimizer AdamW dengan decoupled weight decay & batch kecil)
- **Trial 8:** `alpha=0.75, unfreeze=20%, bs=16, lr=1e-4` (kombinasi model ramping 0.75 dengan fine-tuning)
- **Trial 9:** `opt=adamw, unfreeze=20%, bs=16, lr=2e-4` (fine-tuning 20% layer teratas dengan optimizer AdamW)
- **Trial 10:** `minimalistic=True, unfreeze=20%, lr=1e-4` (fine-tuning pada varian minimalistic untuk pembanding Trial 6)

In [ ]:
import csv
import json
import shutil
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
FOLDER = ROOT / "model-5_mobilenetv3large" if (ROOT / "model-5_mobilenetv3large").exists() else (ROOT / "model-5" if (ROOT / "model-5").exists() else Path("."))
CSV_FILE = FOLDER / "trials.csv"
PNG_FILE = FOLDER / "history_mobilenetv3large.png"
JSON_FILE = FOLDER / "history_oka.json"
riwayat_semua = {}

# Setel True agar 10 trial lama dihapus dan dijalankan ulang bersih dengan normalisasi baru
ULANG_DARI_AWAL = True
if ULANG_DARI_AWAL:
    if CSV_FILE.exists():
        CSV_FILE.unlink()
    if JSON_FILE.exists():
        JSON_FILE.unlink()
WEIGHTS_FILE = Path("/kaggle/working/best_mobilenetv3large.weights.h5") if Path("/kaggle/working").exists() else Path("best_mobilenetv3large.weights.h5")
# Kolom catatan
FIELDNAMES = [
    "trial", "alpha", "minimalistic", "pooling", "lr", "optimizer",
    "unfreeze_pct", "batch_size", "dropout", "val_acc", "val_loss", "best_epoch", "train_time_sec"
]
# 10 Konfigurasi Eksplorasi Parameter (Seed 42)
TRIALS_CONFIG = [
    {"trial": 1,  "alpha": 1.0,  "minimalistic": False, "pooling": "avg", "lr": 1e-3, "optimizer": "adam",  "unfreeze_pct": 0,  "batch_size": 32, "dropout": 0.2},
    {"trial": 2,  "alpha": 1.0,  "minimalistic": False, "pooling": "avg", "lr": 5e-4, "optimizer": "adam",  "unfreeze_pct": 0,  "batch_size": 32, "dropout": 0.3},
    {"trial": 3,  "alpha": 0.75, "minimalistic": False, "pooling": "avg", "lr": 1e-3, "optimizer": "adam",  "unfreeze_pct": 0,  "batch_size": 32, "dropout": 0.2},
    {"trial": 4,  "alpha": 1.0,  "minimalistic": True,  "pooling": "avg", "lr": 1e-3, "optimizer": "adam",  "unfreeze_pct": 0,  "batch_size": 32, "dropout": 0.2},
    {"trial": 5,  "alpha": 1.0,  "minimalistic": False, "pooling": "max", "lr": 5e-4, "optimizer": "adam",  "unfreeze_pct": 0,  "batch_size": 32, "dropout": 0.2},
    {"trial": 6,  "alpha": 1.0,  "minimalistic": False, "pooling": "avg", "lr": 1e-4, "optimizer": "adam",  "unfreeze_pct": 20, "batch_size": 32, "dropout": 0.3},
    {"trial": 7,  "alpha": 1.0,  "minimalistic": False, "pooling": "avg", "lr": 3e-4, "optimizer": "adamw", "unfreeze_pct": 0,  "batch_size": 16, "dropout": 0.2},
    {"trial": 8,  "alpha": 0.75, "minimalistic": False, "pooling": "avg", "lr": 1e-4, "optimizer": "adam",  "unfreeze_pct": 20, "batch_size": 16, "dropout": 0.3},
    {"trial": 9,  "alpha": 1.0,  "minimalistic": False, "pooling": "avg", "lr": 2e-4, "optimizer": "adamw", "unfreeze_pct": 20, "batch_size": 16, "dropout": 0.3},
    {"trial": 10, "alpha": 1.0,  "minimalistic": True,  "pooling": "avg", "lr": 1e-4, "optimizer": "adam",  "unfreeze_pct": 20, "batch_size": 32, "dropout": 0.2},
]
# Siapkan CSV jika belum ada
if not CSV_FILE.exists():
    with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
# Cek trial yang sudah selesai (resume)
selesai_trials = set()
if CSV_FILE.exists() and CSV_FILE.stat().st_size > 0:
    df_check = pd.read_csv(CSV_FILE)
    if "trial" in df_check.columns:
        selesai_trials = set(df_check["trial"].dropna().astype(int).tolist())
best_val_acc = -1.0
best_val_loss = float("inf")
best_history = None
best_cfg = None
for cfg in TRIALS_CONFIG:
    t = cfg["trial"]
    if t in selesai_trials:
        print(f"Trial {t} sudah ada di {CSV_FILE.name}, dilewati.")
        continue
        
    print(f"\n--- Menjalankan Trial {t}/10: alpha={cfg['alpha']}, mini={cfg['minimalistic']}, pool={cfg['pooling']}, lr={cfg['lr']}, opt={cfg['optimizer']}, unfreeze={cfg['unfreeze_pct']}%, bs={cfg['batch_size']}, drop={cfg['dropout']} ---")
    
    t_train_ds, t_val_ds = du.load_data(batch_size=cfg["batch_size"])
    
    model = bangun_model(
        alpha=cfg["alpha"],
        minimalistic=cfg["minimalistic"],
        pooling=cfg["pooling"],
        unfreeze_pct=cfg["unfreeze_pct"],
        dropout=cfg["dropout"]
    )
    
    if cfg["optimizer"] == "adamw":
        opt = keras.optimizers.AdamW(learning_rate=cfg["lr"])
    else:
        opt = keras.optimizers.Adam(learning_rate=cfg["lr"])
        
    model.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
            verbose=1
        )
    ]
    
    start_t = time.time()
    h = model.fit(
        t_train_ds,
        validation_data=t_val_ds,
        epochs=40,
        callbacks=callbacks,
        verbose=1
    )
    durasi = round(time.time() - start_t, 2)
    
    val_losses = h.history["val_loss"]
    val_accs = h.history["val_accuracy"]
    best_idx = int(np.argmin(val_losses))
    best_val_l = round(float(val_losses[best_idx]), 4)
    best_val_a = round(float(val_accs[best_idx]), 4)
    best_ep = best_idx + 1
    
    # LANGSUNG CATAT KE trials.csv
    row = {
        "trial": t,
        "alpha": cfg["alpha"],
        "minimalistic": cfg["minimalistic"],
        "pooling": cfg["pooling"],
        "lr": cfg["lr"],
        "optimizer": cfg["optimizer"],
        "unfreeze_pct": cfg["unfreeze_pct"],
        "batch_size": cfg["batch_size"],
        "dropout": cfg["dropout"],
        "val_acc": best_val_a,
        "val_loss": best_val_l,
        "best_epoch": best_ep,
        "train_time_sec": durasi
    }
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writerow(row)
        f.flush()
        
    riwayat_semua[str(t)] = {k: [round(float(v), 5) for v in vals] for k, vals in h.history.items()}
    with open(JSON_FILE, "w", encoding="utf-8") as f_json:
        json.dump(riwayat_semua, f_json, indent=1)
        
    print(f"Trial {t} selesai -> val_acc: {best_val_a}, val_loss: {best_val_l}, best_epoch: {best_ep}, waktu: {durasi}s")
    
    # Cek trial terbaik
    if (best_val_a > best_val_acc) or (np.isclose(best_val_a, best_val_acc) and best_val_l < best_val_loss):
        best_val_acc = best_val_a
        best_val_loss = best_val_l
        best_history = h.history
        best_cfg = cfg
        model.save_weights(WEIGHTS_FILE)
# Tampilkan seluruh hasil trials.csv
df_hasil = pd.read_csv(CSV_FILE).sort_values(by=["val_acc", "val_loss"], ascending=[False, True])
print("\n=== HASIL TRIALS (Diurutkan dari Terbaik) ===")
print(df_hasil.to_string(index=False))
# Simpan grafik gradient descent dari trial terbaik
if best_history is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(best_history["loss"], label="train_loss")
    ax1.plot(best_history["val_loss"], label="val_loss")
    ax1.set_title(f"Loss (Trial {best_cfg['trial']}: val_acc={best_val_acc}, val_loss={best_val_loss})")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(best_history["accuracy"], label="train_acc")
    ax2.plot(best_history["val_accuracy"], label="val_acc")
    ax2.set_title(f"Akurasi (Trial {best_cfg['trial']})")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(PNG_FILE, dpi=150)
    plt.show()
    print(f"Grafik gradient descent disimpan di {PNG_FILE}")
# Salin trials.csv dan grafik ke /kaggle/working/ agar langsung muncul di tab Output Kaggle
if Path("/kaggle/working").exists():
    shutil.copy2(CSV_FILE, Path("/kaggle/working/trials.csv"))
    if PNG_FILE.exists():
        shutil.copy2(PNG_FILE, Path("/kaggle/working/history_mobilenetv3large.png"))
    if JSON_FILE.exists():
        shutil.copy2(JSON_FILE, Path("/kaggle/working/history_oka.json"))
    print("File trials.csv dan grafik berhasil disalin ke /kaggle/working/ untuk diunduh langsung dari panel Output.")
# Bangun model_terbaik dengan konfigurasi teratas
top_cfg = df_hasil.iloc[0]
model_terbaik = bangun_model(
    alpha=float(top_cfg["alpha"]),
    minimalistic=bool(top_cfg["minimalistic"]),
    pooling=str(top_cfg["pooling"]),
    unfreeze_pct=int(top_cfg["unfreeze_pct"]),
    dropout=float(top_cfg["dropout"])
)
if WEIGHTS_FILE.exists():
    model_terbaik.load_weights(WEIGHTS_FILE)


## 5. Evaluasi test (SEL PALING AKHIR)
Dijalankan setelah 10 trial selesai untuk mengevaluasi model terbaik pada test set (200 gambar).
Hasil test dilaporkan satu kali dan tidak boleh dipakai untuk mengubah hyperparameter.

In [ ]:
# Ubah menjadi True sebelum klik Save & Run All jika ingin seluruh proses (tuning + evaluasi test) selesai otomatis sekaligus
JALANKAN_TEST = True
if JALANKAN_TEST:
    print("=== MENJALANKAN EVALUASI TEST SET (1 KALI) ===")
    test_ds = du.load_test()
    y_true = du.ambil_label(test_ds)
    
    # Prediksi test set
    y_pred_probs = model_terbaik.predict(test_ds)
    y_pred = y_pred_probs.argmax(axis=1)
    
    # Metrik evaluasi
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    test_acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"Akurasi Test: {test_acc:.4f} ({test_acc * 100:.2f}%)")
    print(f"Macro F1    : {macro_f1:.4f}")
    print("\nLaporan Klasifikasi:\n", classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    
    # Parameter & inferensi
    total_params = model_terbaik.count_params()
    start_infer = time.time()
    _ = model_terbaik.predict(test_ds, verbose=0)
    latency = ((time.time() - start_infer) / 200) * 1000
    
    print(f"Jumlah Parameter: {total_params:,}")
    print(f"Waktu Inferensi : {latency:.2f} ms/gambar")
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print(cm)
    
    # Simpan confusion matrix plot beranotasi angka jika di Kaggle
    cm_path = FOLDER / "confusion_matrix_mobilenetv3large.png"
    fig_cm, ax_cm = plt.subplots(figsize=(8, 7))
    im = ax_cm.imshow(cm, cmap="Blues")
    fig_cm.colorbar(im, ax=ax_cm, fraction=0.046, pad=0.04)
    ax_cm.set_xticks(range(len(CLASS_NAMES)))
    ax_cm.set_yticks(range(len(CLASS_NAMES)))
    ax_cm.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=10)
    ax_cm.set_yticklabels(CLASS_NAMES, fontsize=10)
    
    # Tambahkan angka numerik di setiap sel confusion matrix
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            val = cm[i, j]
            color = "white" if val > cm.max() / 2 else "black"
            ax_cm.text(j, i, f"{val}", ha="center", va="center", color=color, fontsize=10, fontweight="bold" if val > 0 else "normal")
            
    ax_cm.set_xlabel("Prediksi", fontsize=12, labelpad=8)
    ax_cm.set_ylabel("Asli", fontsize=12, labelpad=8)
    ax_cm.set_title(f"Confusion Matrix MobileNetV3Large (test, akurasi {test_acc:.1%})", fontsize=13, pad=12)
    plt.tight_layout()
    plt.savefig(cm_path, dpi=150)
    if Path("/kaggle/working").exists():
        shutil.copy2(cm_path, Path("/kaggle/working/confusion_matrix_mobilenetv3large.png"))
    plt.show()
else:
    print("Evaluasi test belum dijalankan (JALANKAN_TEST = False).")
